# Building Model Base Table
This notebook is backbone for understanding of data and assumptions made at the stage of building mbt table.

Purpose: mbt table is ready-to-use for a feautre engineering (encoding, interaction, typecasting, etc) for modeling.

In [228]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

pd.set_option('display.max_columns', None)
warnings.filterwarnings("ignore")

In [229]:
%load_ext autoreload
%autoreload 2

from src.utils.load import load
from src.features import constant, sequence, temporal
from src.targets import target
from src.data.base_table import fill_some_unknown, build_flag

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
df = load("data/canonical/events.parquet")
schema = load("configs/schema.yaml")
profile = load("configs/feature_profile.yaml")

In [303]:
missing_features = df.isna().sum()[df.isna().sum() > 0].index
missing_features
grp_case = df.groupby('case_id')

In [232]:
# percentage of missing values by entries in data
missing_count_by_entries = df.isna().sum().sort_values(ascending=False)
missing_pct_by_entries = (100 * missing_count_by_entries / df.shape[0]).round(2).rename('%missing_by_entries')


# percentage of missing values by cases in data
missing_count_by_cases = df.isna().groupby(df['case_id']).any().sum().sort_values(ascending=False)
missing_pct_by_cases = (100 * missing_count_by_cases / df['case_id'].nunique()).round(2).rename("%missing_by_cases")

missing_pct_record = pd.concat([missing_pct_by_cases, missing_pct_by_entries], axis=1)
missing_pct_record.query("`%missing_by_cases` > 0 or `%missing_by_entries` > 0")

,%missing_by_cases,%missing_by_entries
caused_by_change_id,99.99,99.98
vendor_id,99.94,99.83
asset_id,99.80,99.69
change_request_id,99.61,99.30
root_cause_id,99.04,98.38
assigned_uid,35.15,19.40
reported_symptom,24.58,23.26
assigned_team_gid,10.19,2.95
reported_by_uid,1.01,0.97
resolution_id,0.43,0.50


The first 5 are 99% nulls, for rest we need case level information

caused_by_change_id, vendor_id:
- have 99% missing values, for now they won't be tried for modeling.

asset_id, change_request_id, root_cause_id:
- all these 4 feature are spare, and also have outlier cases with changing values. 
- Later on, flag(presence) of such feature can be tried for modeling.

In [233]:
# handled sparse value with flag 
df.drop(profile['sparse']['columns'], axis=1, inplace=True)

In [234]:
print(f"all {df.columns.size -1} features properties of 24_918 cases at case_level")
grp = df.groupby('case_id')
df_prop = pd.DataFrame(df.dtypes, columns=['dtype']).drop('case_id', axis=0)

df_prop['no_of_changes'] = ((grp.nunique(dropna=True) <= 1).sum() - df.case_id.nunique(dropna=True)).round(2).abs()
df_prop['pct_constant'] = (100* (grp.nunique(dropna=True) <= 1).sum() / df.case_id.nunique(dropna=True)).round(2).abs()

df_prop['all_missing'] = df.isna().groupby(df['case_id']).all().sum()
df_prop['any_missing'] = df.isna().groupby(df['case_id']).any().sum()

df_prop.sort_values(['no_of_changes', 'any_missing'], ascending=True).query('no_of_changes < 10')

all 32 features properties of 24_918 cases at case_level


,dtype,no_of_changes,pct_constant,all_missing,any_missing
opened_at,datetime64[ns],0,100.00,0,0
created_at,datetime64[ns],0,100.00,0,0
notify_email,boolean,0,100.00,0,0
resolved_at,datetime64[ns],0,100.00,0,0
closed_at,datetime64[ns],0,100.00,0,0
created_at_is_imputed,bool,0,100.00,0,0
affected_uid,object,0,100.00,3,3
location_id,object,0,100.00,6,6
resolved_by_uid,object,0,100.00,99,99
resolution_id,object,0,100.00,107,107




contact_channel:
- 5 out of 25k cases making constant (case_leve) feature changing (event_level).
- assumption: intial contact_channel is used for communication and later channel update.
- The later channels might be used for further communication like escalation, sharing info. Since they're only 5 cases, model won't generalize on them.

In [235]:
df_prop.sort_values(['no_of_changes', 'any_missing'], ascending=True).query('no_of_changes > 10')

,dtype,no_of_changes,pct_constant,all_missing,any_missing
used_knowledge_base,bool,208,99.17,0,0
reopen_count,int64,275,98.90,0,0
urgency_level,object,299,98.80,0,0
impact_level,object,315,98.74,0,0
priority_level,object,383,98.46,0,0
category_id,object,1184,95.25,7,7
reported_symptom,object,1323,94.69,5513,6126
subcategory_id,object,1738,93.03,8,8
assigned_uid,object,2858,88.53,658,8759
met_deadline,bool,9114,63.42,0,0


In [236]:
df_prop.sort_values(['any_missing', 'no_of_changes'], ascending=False).query('any_missing > 0')

,dtype,no_of_changes,pct_constant,all_missing,any_missing
assigned_uid,object,2858,88.53,658,8759
reported_symptom,object,1323,94.69,5513,6126
assigned_team_gid,object,9708,61.04,1,2539
reported_by_uid,object,0,100.00,251,251
resolution_id,object,0,100.00,107,107
resolved_by_uid,object,0,100.00,99,99
subcategory_id,object,1738,93.03,8,8
category_id,object,1184,95.25,7,7
location_id,object,0,100.00,6,6
affected_uid,object,0,100.00,3,3


### fix changing record


In [237]:
df['contact_channel'] = grp['contact_channel'].transform('first')

### dropping minor missing cases


In [238]:
missing_table = df.isna().groupby(df['case_id']).all()
missing_count = missing_table.sum(axis=0)
feature_names = missing_count[(missing_count > 0) & (missing_count < 10)].index

cases_to_drop = df.isna().groupby(df['case_id']).all()[feature_names].any(axis=1)
drop_ids = cases_to_drop[cases_to_drop].index

df = df[~df['case_id'].isin(drop_ids)]

### fixing major (1000s) missing cases

In [286]:
missing_table = df.isna().groupby(df['case_id']).any()
missing_count = missing_table.sum(axis=0)
feature_names = missing_count[(missing_count > 1000)].index
feature_names

Index(['reported_symptom', 'assigned_team_gid', 'assigned_uid'], dtype='object')

In [287]:
df[feature_names].isna().sum()

reported_symptom     32927
assigned_team_gid     4177
assigned_uid         27467
dtype: int64

In [292]:
df[['reported_symptom', 'assigned_team_gid', 'assigned_uid']].isna().groupby(df['case_id']).any().sum()

reported_symptom     6117
assigned_team_gid    2534
assigned_uid         8751
dtype: int64

In [290]:
df[feature_names].nunique()

reported_symptom     525
assigned_team_gid     78
assigned_uid         233
dtype: int64

In [288]:
df['reopen_count'].unique()

array([0, 1, 2, 3, 4, 5, 6, 7, 8])

In [283]:
df.groupby(['case_id', 'reopen_count', 'reassignment_count'])['assigned_uid'].nunique(dropna=True).value_counts()

assigned_uid
1    38529
0     8366
2     1683
3       58
4        7
5        1
Name: count, dtype: int64

In [284]:
df.groupby(['case_id', 'reopen_count', 'reassignment_count'])['assigned_team_gid'].nunique(dropna=True).value_counts()

assigned_team_gid
1    47487
0     1121
2       34
3        2
Name: count, dtype: int64

'reported_symptom', 'assigned_team_gid', 'assigned_uid' are not highly missing values, and no valid product logic (reassignment/reopen count, features nmi) is found. 

- Forward or backward filling on assigned team or agent id will fabricate their pattern for model learning.
- reported_symptom is not a proper low cardinality (2% of the case), missing_flag likely to work best for this after forward. Assuming once the user has provided symptom/perception of issue it is known to the system.

In [304]:
df['reported_symptom_missing_flag'] = build_flag(df['reported_symptom'], missing_flags=False) # capture original missingness
df['reported_symptom'] = grp_case['reported_symptom'].transform('ffill')
df['reported_symptom'] = df['reported_symptom'].fillna('Unknown')

In [306]:
df['assigned_uid'] = df['assigned_uid'].fillna('Unknown')
df['assigned_team_gid'] = df['assigned_team_gid'].fillna('Unknown')

### features missing 100s of cases

In [309]:
missing_table = df.isna().groupby(df['case_id']).all()
missing_count = missing_table.sum(axis=0)
feature_names = missing_count[(missing_count > 10) & (missing_count < 1000)].index.tolist()
feature_names

['reported_by_uid', 'resolution_id', 'resolved_by_uid']

resolution_id and resolved_by_uid columns will cause leakage and are forbidden as model learning feature. 

In [310]:
df[feature_names].nunique()

reported_by_uid    208
resolution_id       17
resolved_by_uid    215
dtype: int64

reported_by_uid and assigned_uid are low-medium cardinality (200s in 25k cases) and uid does have any ordering. 

In [312]:
df[['affected_uid', 'updated_by_uid']].nunique() < 0.015 * df.case_id.nunique()

affected_uid      False
updated_by_uid    False
dtype: bool

In [314]:
print(fill_some_unknown(df)['reported_by_uid'].isna().sum(),
fill_some_unknown(df)['reported_by_uid'].unique().__contains__('Unknown')
)

0 True


In [315]:
df = fill_some_unknown(df)

In [316]:
print(df['reported_symptom'].isna().sum(),
      df['reported_symptom'].nunique())

0 526


In [317]:
df['reported_symptom_flag'] = build_flag(df['reported_symptom'], missing_flags=False)

### handling minor shift within cases.

In [322]:
print(f"all {df.columns.size -1} features properties of 24_918 cases at case_level")
grp = df.groupby('case_id')
df_prop = pd.DataFrame(df.dtypes, columns=['dtype']).drop('case_id', axis=0)

df_prop['no_of_changes'] = ((grp.nunique(dropna=True) <= 1).sum() - df.case_id.nunique(dropna=True)).round(2).abs()
df_prop['pct_constant'] = (100* (grp.nunique(dropna=True) <= 1).sum() / df.case_id.nunique(dropna=True)).round(2).abs()

df_prop['all_missing'] = df.isna().groupby(df['case_id']).all().sum()
df_prop['any_missing'] = df.isna().groupby(df['case_id']).any().sum()

df_prop.sort_values(['no_of_changes', 'any_missing'], ascending=True).query('no_of_changes > 0 and pct_constant > 50')


all 34 features properties of 24_918 cases at case_level


,dtype,no_of_changes,pct_constant,all_missing,any_missing
used_knowledge_base,bool,208,99.16,0,0
reopen_count,int64,275,98.90,0,0
urgency_level,object,299,98.80,0,0
impact_level,object,315,98.74,0,0
priority_level,object,383,98.46,0,0
reported_symptom_missing_flag,int64,612,97.54,0,0
category_id,object,1182,95.25,0,0
reported_symptom,object,1531,93.85,0,0
subcategory_id,object,1737,93.03,0,0
assigned_uid,object,8947,64.08,0,0


features are likely genuine change and real signal:
- used_knowledge_base: ex, first knowledge not used, later accessed for further investigation.
- reopen_count: only 1% cases are reopened. 
- priority_level: ex, user raised the urgency later to reduce deadline; agent changed impact upon case investigation. 

similarly senarios can be listed for others.

### inspect transition of feature with 5% changing instances.

In [336]:
df.groupby('case_id')['used_knowledge_base'].nunique()[df.groupby('case_id')['used_knowledge_base'].nunique() > 1]

case_id
INC0000298    2
INC0000302    2
INC0000307    2
INC0000324    2
INC0000343    2
             ..
INC0031839    2
INC0032041    2
INC0032224    2
INC0032593    2
INC0052223    2
Name: used_knowledge_base, Length: 208, dtype: int64

In [331]:
def summarize_transitions(df, col):
    return (
        df
        .groupby('case_id')[col]
        .apply(lambda x: f"{x.iloc[0]} → {x.iloc[-1]}" if x.nunique() > 1 else None)
        .dropna()
        .value_counts()
    )

In [332]:
summarize_transitions(df, 'used_knowledge_base')

used_knowledge_base
True → False    106
False → True    102
Name: count, dtype: int64

In [329]:
summarize_transitions(df, 'reopen_count')


reopen_count
0 → 1    245
0 → 2     19
0 → 4      4
0 → 3      4
0 → 6      2
0 → 8      1
Name: count, dtype: int64